In [1]:
import random
import os
import pandas as pd

from sklearn.model_selection import KFold, StratifiedKFold

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, BertModel, BertConfig, AutoConfig
import os
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score,f1_score, classification_report,recall_score, precision_score
import numpy as np
from sklearn.model_selection import train_test_split


In [2]:
#CONFIG

#General 
CATEG = 'pc'
MODEL_OUT_DIR = 'model/' #Path to save model and files
RESULTS_DATA  = 'TodosBanco.csv' #Path Test Data
MODEL_NAME = 'neuralmind/bert-base-portuguese-cased' #Model name


#Device 
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 128
MAX_LEN = 122


In [3]:
dados_resultado = pd.read_csv(RESULTS_DATA)
dados_resultado = dados_resultado.loc[:,['index_banco','Datetime', 'Username', 'Text']]

In [4]:
seed_val = 22
random.seed(seed_val)
np.random.seed(seed_val)
torch.manual_seed(seed_val)
torch.cuda.manual_seed_all(seed_val)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME,max_lenght=MAX_LEN)

class GerarDataset(Dataset):
  def __init__(self, data, tokenizer):
        self.df = data.reset_index()
        self.tokenizer = tokenizer

  def __len__(self):
        return self.df.shape[0]

  def __getitem__(self, index):
        #Select the sentence and label at the specified index in the data frame
        texto_tratado = self.df.loc[index, 'Text']
        id = self.df.loc[index, 'index_banco']
        if pd.isna(texto_tratado):
          texto_tratado = ""    
        #target = self.df.loc[index, CATEG]
        #identifier = self.df.loc[index, 'id']
        tokens = tokenizer(texto_tratado,
                           padding='max_length',
                           max_length=MAX_LEN,
                           add_special_tokens=True,
                           return_tensors='pt',
                           truncation=True)
        
        input_ids = tokens["input_ids"].clone().detach()
        attention_mask = tokens["attention_mask"].clone().detach()
        #target = torch.tensor(target, dtype=torch.long)
        #torch.tensor(target, dtype=torch.long)
        return input_ids, attention_mask, id


model_preds_list = []


In [5]:
def GerarResultado(nome_modelo):
    
    caminho_modelo = os.path.join("model",nome_modelo)

    #Modelo 
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)

    # Tokenizando e transformando em inputs (att masks, labels, etc.)
    geral_data = GerarDataset(dados_resultado,tokenizer)
    geral_loader = DataLoader(geral_data, batch_size=BATCH_SIZE, shuffle=True)

    model.load_state_dict(torch.load(caminho_modelo))
    model.to(DEVICE)
    model.eval()
    y_real, y_pred = [], []  # Armazenam as previsões e os rótulos reais
    ids_predicoes = []

    with torch.no_grad():
        for i, (input_ids, attention_mask, ids) in enumerate(iterable=geral_loader):
            input_ids, attention_mask = input_ids.to(DEVICE), attention_mask.to(DEVICE)

            input_ids = input_ids.squeeze(1)  # Remove a dimensão extra
            attention_mask = attention_mask.squeeze(1)  # Remove a dimensão extra
            
            output = model(input_ids=input_ids, attention_mask=attention_mask)
         
            preditos = torch.argmax(output.logits, 1).to("cpu").tolist()
            y_pred.extend(preditos)  # Armazena as previsões
            ids_predicoes.extend(ids.tolist())
            
            
      # Converte as listas para numpy arrays para calcular as métricas
    #y_real = np.array(y_real)
    y_pred = np.array(y_pred)
   

    # Calcula as principais métricas de classificação
    
    
    return  ids_predicoes, y_pred

In [6]:
modelos = [i for i in  os.listdir("model/") if i.endswith('bin') and CATEG.lower() in i]

In [7]:
modelos

In [8]:
vals_f1, acc_validacao, precisao_validacao, recall_validacao = [], [], [], []
f1s_class1, precisions_class1, recalls_class1,nomes_modelos = [], [], [],[]
resultados = []
ids_predicoes, y_real, y_pred = [], [], []
#modelos = [i for i in  os.listdir("model/") if i.endswith('bin') and CATEG.lower() in i]

nome_modelo = 'aMODELO06_pc__fold_0_epoch_2.bin'

ids_predicoes,  y_pred  = GerarResultado(nome_modelo) 

previsoes = pd.DataFrame({
    'index_banco': ids_predicoes,
    CATEG: y_pred
})


In [9]:
dados_resultado.head()

In [10]:
#dados_resultado.merge(previsoes,on="index_banco").to_excel("Resultados_AE_modelo6_fold2_epoca4.xlsx",index=False)
#dados_resultado.merge(previsoes,on="index_banco").to_excel("Resultados_PC_modelo4_fold1_epoca4.xlsx",index=False)
#dados_resultado.merge(previsoes,on="index_banco").to_excel("Resultados_PC_modelo6_fold0_epoca2.xlsx",index=False)
